In [1]:
%pip install requests beautifulsoup4 lxml pandas numpy tqdm sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import requests
import numpy as np
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
import faiss

print("RAG environment ready!")

c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAG environment ready!


In [3]:
import requests
import bs4
import lxml
import pandas
import numpy
import tqdm
import sentence_transformers
import faiss

print("requests:", requests.__version__)
print("beautifulsoup4:", bs4.__version__)
print("lxml:", lxml.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("FAISS: imported successfully")

print("\n✅ All RAG dependencies are ready!")

requests: 2.34.2
beautifulsoup4: 4.15.0
lxml: 6.1.3
pandas: 3.0.6
numpy: 2.4.6
sentence-transformers: 6.1.0
FAISS: imported successfully

✅ All RAG dependencies are ready!


In [4]:
import json
import os

rag_sources = [
    {
        "id": "radiologyinfo_chest_xray_report",
        "title": "How to Read Your Chest X-ray Report",
        "organization": "RadiologyInfo",
        "url": "https://www.radiologyinfo.org/en/info/article-chest-xray-report",
        "type": "medical_reference"
    },
    {
        "id": "chestxray14_research",
        "title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
        "organization": "NIH / Research Literature",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "type": "research"
    }
]

os.makedirs("rag/metadata", exist_ok=True)

with open(
    "rag/metadata/sources.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(rag_sources, f, indent=4)

print("RAG source registry created!")
print("Sources:", len(rag_sources))

for source in rag_sources:
    print(f"✓ {source['title']}")

RAG source registry created!
Sources: 2
✓ How to Read Your Chest X-ray Report
✓ ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification


In [5]:
import requests

pmc_id = "PMC9131331"

url = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    f"efetch.fcgi?db=pmc&id={pmc_id}&retmode=xml"
)

response = requests.get(url, timeout=60)

print("HTTP status:", response.status_code)
print("Response size:", len(response.text), "characters")
print("Starts with:", response.text[:100])


HTTP status: 200
Response size: 11528 characters
Starts with: <?xml version="1.0"  ?><!DOCTYPE pmc-articleset PUBLIC "-//NLM//DTD ARTICLE SET 2.0//EN" "https://dt


In [6]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "xml")

# Extract the article text
text_parts = []

for element in soup.find_all(["article-title", "abstract", "p"]):
    text = element.get_text(" ", strip=True)
    if text:
        text_parts.append(text)

article_text = "\n\n".join(text_parts)

print("Extracted article text:", len(article_text), "characters")
print("\nFirst 2000 characters:\n")
print(article_text[:2000])

Extracted article text: 4637 characters

First 2000 characters:

Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Contributions: (I) Conception and design: All authors; (II) Administrative support: All authors; (III) Provision of study materials or patients: C Chen, S Yang; (IV) Collection and assembly of data: F Li; (V) Data analysis and interpretation: F Li; (VI) Manuscript writing: All authors; (VII) Final approval of manuscript: All authors.

These authors contributed equally to this work and should be considered as co-first authors.

Conflicts of Interest: All authors have completed the ICMJE uniform disclosure form (available at https://qims.amegroups.com/article/view/10.21037/qims-21-1117/coif ). FL has a pending patent (No. 202110640995.8). The other authors have no conflicts of interest to declare.

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to 

In [8]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nContents:")
print(os.listdir())

Current working directory:
c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\notebooks

Contents:
['Chest_Xray_AI.ipynb', 'rag', 'RAG_LLM.ipynb']


In [10]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

RAG_DIR = PROJECT_ROOT / "rag"
DOCUMENTS_DIR = RAG_DIR / "documents"
CHUNKS_DIR = RAG_DIR / "chunks"
VECTOR_STORE_DIR = RAG_DIR / "vector_store"
METADATA_DIR = RAG_DIR / "metadata"

for directory in [
    RAG_DIR,
    DOCUMENTS_DIR,
    CHUNKS_DIR,
    VECTOR_STORE_DIR,
    METADATA_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("RAG directory:", RAG_DIR)
print("Documents directory:", DOCUMENTS_DIR)
print("✓ RAG paths ready!")

Project root: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI
RAG directory: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag
Documents directory: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents
✓ RAG paths ready!


In [11]:
document_path = DOCUMENTS_DIR / "chestxray14_research.txt"

with open(
    document_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(article_text)

print("Document saved successfully!")
print("Characters:", len(article_text))
print("Path:", document_path)

Document saved successfully!
Characters: 4637
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\chestxray14_research.txt


In [12]:
# Inspect the document structure

with open(
    document_path,
    "r",
    encoding="utf-8"
) as f:
    text = f.read()

paragraphs = [
    p.strip()
    for p in text.split("\n\n")
    if p.strip()
]

print("Total characters:", len(text))
print("Total paragraphs:", len(paragraphs))

print("\nFirst 10 paragraphs:\n")

for i, paragraph in enumerate(paragraphs[:10]):
    print(f"\n--- Paragraph {i + 1} ---")
    print(paragraph[:500])

Total characters: 4637
Total paragraphs: 9

First 10 paragraphs:


--- Paragraph 1 ---
Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

--- Paragraph 2 ---
Contributions: (I) Conception and design: All authors; (II) Administrative support: All authors; (III) Provision of study materials or patients: C Chen, S Yang; (IV) Collection and assembly of data: F Li; (V) Data analysis and interpretation: F Li; (VI) Manuscript writing: All authors; (VII) Final approval of manuscript: All authors.

--- Paragraph 3 ---
These authors contributed equally to this work and should be considered as co-first authors.

--- Paragraph 4 ---
Conflicts of Interest: All authors have completed the ICMJE uniform disclosure form (available at https://qims.amegroups.com/article/view/10.21037/qims-21-1117/coif ). FL has a pending patent (No. 202110640995.8). The other authors have no conflicts of interest to declare.

--- Paragraph 5 ---
Background Compu

In [13]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "xml")

title = soup.find("article-title")
title_text = title.get_text(" ", strip=True) if title else ""

abstract = soup.find("abstract")
abstract_text = abstract.get_text(" ", strip=True) if abstract else ""

# Extract paragraph-level content only
paragraphs = []

for p in soup.find_all("p"):
    text = p.get_text(" ", strip=True)
    if text:
        paragraphs.append(text)

# Build clean document
clean_article_text = "\n\n".join(
    [title_text, abstract_text] + paragraphs
)

print("Cleaned document characters:", len(clean_article_text))
print("Paragraphs:", len(paragraphs))

print("\nFirst 2000 characters:\n")
print(clean_article_text[:2000])

Cleaned document characters: 4637
Paragraphs: 7

First 2000 characters:

Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient for medical image detection. To address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for modeling patch-wise and inter-patch dependencies. Methods We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the tra

In [14]:
document_path = DOCUMENTS_DIR / "chestxray14_research.txt"

with open(
    document_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(clean_article_text)

print("Clean document saved!")
print("Characters:", len(clean_article_text))
print("Path:", document_path)

Clean document saved!
Characters: 4637
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\chestxray14_research.txt


In [15]:
chunk_size = 800
chunk_overlap = 150

chunks = []

start = 0
text_length = len(clean_article_text)

while start < text_length:
    end = min(start + chunk_size, text_length)

    chunk = clean_article_text[start:end].strip()

    if chunk:
        chunks.append(chunk)

    start += chunk_size - chunk_overlap

print("Document characters:", text_length)
print("Number of chunks:", len(chunks))

print("\nChunk sizes:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}: {len(chunk)} characters")

print("\nFirst chunk:\n")
print(chunks[0])

Document characters: 4637
Number of chunks: 8

Chunk sizes:
Chunk 1: 800 characters
Chunk 2: 800 characters
Chunk 3: 799 characters
Chunk 4: 799 characters
Chunk 5: 800 characters
Chunk 6: 799 characters
Chunk 7: 737 characters
Chunk 8: 87 characters

First chunk:

Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient for medical image detection. To address this problem, in this paper, we propose a CXR detection method which integ